In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!find /content/drive/MyDrive -name "archive (1).zip"

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Skincancer_project/archive (1).zip"
extract_path = "/content/ham10000"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
!find /content/ham10000 -maxdepth 2 -type f | head -30

In [ ]:
!find /content/ham10000 -maxdepth 2 -type d

In [ ]:
import os

for root, dirs, files in os.walk("/content/ham10000"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
import pandas as pd

metadata_path = "/content/ham10000/HAM10000_metadata.csv"

df = pd.read_csv(metadata_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

In [ ]:
print(df["dx"].value_counts())


In [ ]:
print(df[["dx", "age", "sex"]].isnull().sum())

In [ ]:
print("Age statistics:")
print(df["age"].describe())

print("\nGender distribution:")
print(df["sex"].value_counts(dropna=False))

In [ ]:
# Create binary cancer label

malignant_classes = ['mel', 'bcc', 'akiec']

df['binary_label'] = df['dx'].apply(
    lambda x: 1 if x in malignant_classes else 0
)

# 0 = Benign
# 1 = Malignant

print(df[['dx', 'binary_label']].drop_duplicates().sort_values('dx'))

print("\nBinary class distribution:")
print(df['binary_label'].value_counts())

print("\nPercentage:")
print(df['binary_label'].value_counts(normalize=True) * 100)

In [ ]:
pd.crosstab(df['dx'], df['binary_label'])

In [ ]:
median_age = df['age'].median()

df['age'] = df['age'].fillna(median_age)

print("Missing ages after imputation:", df['age'].isnull().sum())
print("Median age used:", median_age)

In [ ]:
print(df['sex'].value_counts())

In [ ]:
df['gender_encoded'] = df['sex'].map({
    'male': 0,
    'female': 1,
    'unknown': 2
})

print(df[['sex', 'gender_encoded']].drop_duplicates())

In [ ]:
import os

IMAGE_DIR_1 = "/content/ham10000/HAM10000_images_part_1"
IMAGE_DIR_2 = "/content/ham10000/HAM10000_images_part_2"

def find_image(image_id):
    path1 = os.path.join(IMAGE_DIR_1, image_id + ".jpg")
    path2 = os.path.join(IMAGE_DIR_2, image_id + ".jpg")

    if os.path.exists(path1):
        return path1
    elif os.path.exists(path2):
        return path2
    return None

df["image_path"] = df["image_id"].apply(find_image)

print("Total records:", len(df))
print("Images found:", df["image_path"].notna().sum())
print("Images missing:", df["image_path"].isna().sum())

In [ ]:
print("Total images:", len(df))
print("Unique lesions:", df["lesion_id"].nunique())
print("Images belonging to repeated lesions:",
      df["lesion_id"].duplicated(keep=False).sum())

test spliting

Image preprocessing

In [ ]:
import matplotlib.pyplot as plt

# Class counts
labels = ['Benign', 'Malignant']
sizes = [8061, 1954]

# Create pie chart
plt.figure(figsize=(7, 7))

plt.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90
)

plt.title('HAM10000 Dataset: Benign vs Malignant')
plt.axis('equal')

plt.show()

In [ ]:
import matplotlib.pyplot as plt

labels = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']

sizes = [
    6705,
    1113,
    1099,
    514,
    327,
    142,
    115
]

plt.figure(figsize=(8, 8))

plt.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90
)

plt.title('HAM10000 Dataset - Lesion Type Distribution')
plt.axis('equal')

plt.show()

tseting whether efficient data split

In [ ]:
!pip install iterative-stratification -q

In [ ]:
import pandas as pd
import numpy as np

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Load metadata
metadata_path = "/content/ham10000/HAM10000_metadata.csv"
df = pd.read_csv(metadata_path)

print("Total images:", len(df))
print("Unique lesions:", df["lesion_id"].nunique())

In [ ]:
# Create lesion-level dataframe
lesion_df = (
    df.groupby("lesion_id")
      .agg({
          "dx": "first"
      })
      .reset_index()
)

print("Lesion-level records:", len(lesion_df))
print(lesion_df["dx"].value_counts())

In [ ]:
# One-hot encode diagnosis
lesion_labels = pd.get_dummies(lesion_df["dx"])

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    msss.split(lesion_df, lesion_labels)
)

train_lesion_ids = set(
    lesion_df.iloc[train_idx]["lesion_id"]
)

temp_lesion_ids = set(
    lesion_df.iloc[temp_idx]["lesion_id"]
)

print("Train lesions:", len(train_lesion_ids))
print("Temporary lesions:", len(temp_lesion_ids))

In [ ]:
temp_df = lesion_df[
    lesion_df["lesion_id"].isin(temp_lesion_ids)
].reset_index(drop=True)

temp_labels = pd.get_dummies(temp_df["dx"])

msss2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    msss2.split(temp_df, temp_labels)
)

val_lesion_ids = set(
    temp_df.iloc[val_idx]["lesion_id"]
)

test_lesion_ids = set(
    temp_df.iloc[test_idx]["lesion_id"]
)

print("Validation lesions:", len(val_lesion_ids))
print("Test lesions:", len(test_lesion_ids))

In [ ]:
train_df = df[
    df["lesion_id"].isin(train_lesion_ids)
].copy()

val_df = df[
    df["lesion_id"].isin(val_lesion_ids)
].copy()

test_df = df[
    df["lesion_id"].isin(test_lesion_ids)
].copy()

print("\nImage-level split:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
train_set = set(train_df["lesion_id"])
val_set = set(val_df["lesion_id"])
test_set = set(test_df["lesion_id"])

print("\nOverlap check:")
print("Train-Test:", len(train_set & test_set))
print("Train-Validation:", len(train_set & val_set))
print("Validation-Test:", len(val_set & test_set))

In [ ]:
print("\nTRAIN distribution:")
print(train_df["dx"].value_counts(normalize=True).round(3))

print("\nVALIDATION distribution:")
print(val_df["dx"].value_counts(normalize=True).round(3))

print("\nTEST distribution:")
print(test_df["dx"].value_counts(normalize=True).round(3))

In [ ]:
# 7-class lesion mapping
class_names = ['mel', 'nv', 'bkl', 'bcc', 'akiec', 'df', 'vasc']

class_to_idx = {
    name: idx for idx, name in enumerate(class_names)
}

# Add lesion labels
train_df['class_label'] = train_df['dx'].map(class_to_idx)
val_df['class_label'] = val_df['dx'].map(class_to_idx)
test_df['class_label'] = test_df['dx'].map(class_to_idx)


# Benign / Malignant mapping
benign_classes = ['nv', 'bkl', 'df', 'vasc']
malignant_classes = ['mel', 'bcc', 'akiec']

train_df['binary_label'] = train_df['dx'].apply(
    lambda x: 0 if x in benign_classes else 1
)

val_df['binary_label'] = val_df['dx'].apply(
    lambda x: 0 if x in benign_classes else 1
)

test_df['binary_label'] = test_df['dx'].apply(
    lambda x: 0 if x in benign_classes else 1
)

print("Labels created successfully!")

print("\nTrain:")
print(train_df[['dx', 'class_label', 'binary_label']].head())

print("\nTest:")
print(test_df[['dx', 'class_label', 'binary_label']].head())

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import glob
import torch
from torchvision import transforms

# ==============================
# TRANSFORMS
# ==============================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ==============================
# DATASET
# ==============================

class SkinDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_id = row["image_id"]

        # Search inside both HAM10000 image folders
        image_paths = glob.glob(
            f"/content/ham10000/**/{image_id}.jpg",
            recursive=True
        )

        if len(image_paths) == 0:
            raise FileNotFoundError(
                f"Image not found: {image_id}"
            )

        image = Image.open(image_paths[0]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        lesion_label = torch.tensor(
            row["class_label"],
            dtype=torch.long
        )

        binary_label = torch.tensor(
            row["binary_label"],
            dtype=torch.long
        )

        return image, lesion_label, binary_label


# ==============================
# CREATE DATASETS
# ==============================

train_dataset = SkinDataset(
    train_df,
    transform=train_transform
)

val_dataset = SkinDataset(
    val_df,
    transform=val_test_transform
)

test_dataset = SkinDataset(
    test_df,
    transform=val_test_transform
)


# ==============================
# CREATE DATALOADERS
# ==============================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)


print("DataLoaders created successfully!")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
images, lesion_labels, binary_labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Lesion labels shape:", lesion_labels.shape)
print("Binary labels shape:", binary_labels.shape)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load pretrained EfficientNet-B0
backbone_eff = models.efficientnet_b0(weights="DEFAULT")

# Multi-task EfficientNet-B0
class MultiTaskEfficientNet(nn.Module):
    def __init__(self, backbone):
        super().__init__()

        # Feature extractor
        self.features = backbone.features

        # Number of extracted features
        num_features = backbone.classifier[1].in_features

        # Two output heads
        self.lesion_head = nn.Linear(num_features, 7)
        self.binary_head = nn.Linear(num_features, 2)

    def forward(self, x):

        # Extract image features
        x = self.features(x)

        # Global average pooling
        x = torch.nn.functional.adaptive_avg_pool2d(x, 1)

        # Flatten
        x = torch.flatten(x, 1)

        # Predictions
        lesion_output = self.lesion_head(x)
        binary_output = self.binary_head(x)

        return lesion_output, binary_output


# Create model
model_eff = MultiTaskEfficientNet(backbone_eff).to(device)

print("EfficientNet-B0 created successfully!")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 7-class lesion weights
lesion_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(7),
    y=train_df["class_label"].values
)

lesion_weights = torch.tensor(
    lesion_weights,
    dtype=torch.float32
).to(device)

print("Lesion class weights:")

for name, weight in zip(class_names, lesion_weights):
    print(f"{name}: {weight:.3f}")


# Binary Benign/Malignant weights
binary_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["binary_label"].values
)

binary_weights = torch.tensor(
    binary_weights,
    dtype=torch.float32
).to(device)

print("\nBinary class weights:")
print("Benign:", binary_weights[0].item())
print("Malignant:", binary_weights[1].item())

In [ ]:
# Loss for 7-class lesion classification
lesion_criterion = nn.CrossEntropyLoss(
    weight=lesion_weights
)

# Loss for Benign/Malignant classification
binary_criterion = nn.CrossEntropyLoss(
    weight=binary_weights
)


# Combined multi-task loss
def multitask_loss(
    lesion_output,
    binary_output,
    lesion_target,
    binary_target
):

    lesion_loss = lesion_criterion(
        lesion_output,
        lesion_target
    )

    binary_loss = binary_criterion(
        binary_output,
        binary_target
    )

    total_loss = lesion_loss + binary_loss

    return total_loss, lesion_loss, binary_loss


print("Loss functions created successfully!")

In [ ]:
optimizer_eff = torch.optim.Adam(
    model_eff.parameters(),
    lr=0.0001,
    weight_decay=1e-4
)

print("Optimizer created successfully!")

In [ ]:
model_eff.eval()

with torch.no_grad():

    images, lesion_targets, binary_targets = next(iter(train_loader))

    images = images.to(device)
    lesion_targets = lesion_targets.to(device)
    binary_targets = binary_targets.to(device)

    lesion_output, binary_output = model_eff(images)

print("Image batch:", images.shape)
print("Lesion output:", lesion_output.shape)
print("Binary output:", binary_output.shape)

In [ ]:
num_epochs = 10

best_val_binary_acc = 0.0

train_losses = []
val_losses = []
train_binary_accs = []
val_binary_accs = []

for epoch in range(num_epochs):

    # =========================
    # TRAINING
    # =========================
    model_eff.train()

    running_loss = 0.0
    correct_binary = 0
    total_binary = 0

    for images, lesion_targets, binary_targets in train_loader:

        images = images.to(device)
        lesion_targets = lesion_targets.to(device)
        binary_targets = binary_targets.to(device)

        # Clear gradients
        optimizer_eff.zero_grad()

        # Forward pass
        lesion_output, binary_output = model_eff(images)

        # Calculate loss
        total_loss, lesion_loss, binary_loss = multitask_loss(
            lesion_output,
            binary_output,
            lesion_targets,
            binary_targets
        )

        # Backpropagation
        total_loss.backward()

        # Update weights
        optimizer_eff.step()

        # Statistics
        running_loss += total_loss.item()

        binary_predictions = torch.argmax(binary_output, dim=1)

        correct_binary += (
            binary_predictions == binary_targets
        ).sum().item()

        total_binary += binary_targets.size(0)

    train_loss = running_loss / len(train_loader)
    train_binary_acc = correct_binary / total_binary


    # =========================
    # VALIDATION
    # =========================
    model_eff.eval()

    val_running_loss = 0.0
    val_correct_binary = 0
    val_total_binary = 0

    with torch.no_grad():

        for images, lesion_targets, binary_targets in val_loader:

            images = images.to(device)
            lesion_targets = lesion_targets.to(device)
            binary_targets = binary_targets.to(device)

            lesion_output, binary_output = model_eff(images)

            total_loss, lesion_loss, binary_loss = multitask_loss(
                lesion_output,
                binary_output,
                lesion_targets,
                binary_targets
            )

            val_running_loss += total_loss.item()

            binary_predictions = torch.argmax(
                binary_output,
                dim=1
            )

            val_correct_binary += (
                binary_predictions == binary_targets
            ).sum().item()

            val_total_binary += binary_targets.size(0)

    val_loss = val_running_loss / len(val_loader)
    val_binary_acc = val_correct_binary / val_total_binary


    # Save history
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_binary_accs.append(train_binary_acc)
    val_binary_accs.append(val_binary_acc)


    # =========================
    # SAVE BEST MODEL
    # =========================
    if val_binary_acc > best_val_binary_acc:

        best_val_binary_acc = val_binary_acc

        torch.save(
            model_eff.state_dict(),
            "/content/drive/MyDrive/Skincancer_project/best_efficientnet_final.pth"
        )

        print("Best model saved!")


    # =========================
    # PRINT RESULTS
    # =========================
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Binary Acc: {train_binary_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Binary Acc: {val_binary_acc:.4f}"
    )


print("\nTraining completed!")
print("Best Validation Binary Accuracy:",
      f"{best_val_binary_acc:.4f}")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Load the saved best EfficientNet-B0 checkpoint

checkpoint_path = "/content/drive/MyDrive/Skincancer_project/best_efficientnet_final.pth"

model_eff.load_state_dict(
    torch.load(checkpoint_path, map_location=device)
)

model_eff = model_eff.to(device)
model_eff.train()

print("✅ Saved EfficientNet-B0 checkpoint loaded successfully!")
print("Starting from the saved Epoch 7 model.")

In [ ]:
optimizer_eff = torch.optim.Adam(
    model_eff.parameters(),
    lr=0.0001,
    weight_decay=1e-4
)

print("✅ Optimizer ready!")

In [ ]:
# Continue training from Epoch 7 → Epoch 10

best_val_binary_acc = 0.8780   # Epoch 7 best validation accuracy

for epoch in range(8, 11):

    # =========================
    # TRAIN
    # =========================
    model_eff.train()

    train_total_loss = 0
    train_binary_correct = 0
    train_total = 0

    for images, lesion_targets, binary_targets in train_loader:

        images = images.to(device)
        lesion_targets = lesion_targets.to(device)
        binary_targets = binary_targets.to(device)

        optimizer_eff.zero_grad()

        lesion_output, binary_output = model_eff(images)

        total_loss, lesion_loss, binary_loss = multitask_loss(
            lesion_output,
            binary_output,
            lesion_targets,
            binary_targets
        )

        total_loss.backward()
        optimizer_eff.step()

        train_total_loss += total_loss.item() * images.size(0)

        binary_pred = torch.argmax(binary_output, dim=1)

        train_binary_correct += (
            binary_pred == binary_targets
        ).sum().item()

        train_total += images.size(0)

    train_loss = train_total_loss / train_total
    train_binary_acc = train_binary_correct / train_total


    # =========================
    # VALIDATION
    # =========================
    model_eff.eval()

    val_total_loss = 0
    val_binary_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, lesion_targets, binary_targets in val_loader:

            images = images.to(device)
            lesion_targets = lesion_targets.to(device)
            binary_targets = binary_targets.to(device)

            lesion_output, binary_output = model_eff(images)

            total_loss, lesion_loss, binary_loss = multitask_loss(
                lesion_output,
                binary_output,
                lesion_targets,
                binary_targets
            )

            val_total_loss += total_loss.item() * images.size(0)

            binary_pred = torch.argmax(binary_output, dim=1)

            val_binary_correct += (
                binary_pred == binary_targets
            ).sum().item()

            val_total += images.size(0)

    val_loss = val_total_loss / val_total
    val_binary_acc = val_binary_correct / val_total


    # =========================
    # RESULTS
    # =========================
    print(
        f"Epoch [{epoch}/10] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Binary Acc: {train_binary_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Binary Acc: {val_binary_acc:.4f}"
    )


    # =========================
    # SAVE ONLY IF BETTER
    # =========================
    if val_binary_acc > best_val_binary_acc:

        best_val_binary_acc = val_binary_acc

        torch.save(
            model_eff.state_dict(),
            "/content/drive/MyDrive/Skincancer_project/"
            "best_efficientnet_final.pth"
        )

        print("✅ New best model saved!")

    else:
        print("No improvement - Epoch 7/best model kept.")

In [ ]:
# Load the best EfficientNet-B0 model (Epoch 9)

best_model_path = "/content/drive/MyDrive/Skincancer_project/best_efficientnet_final.pth"

model_eff.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

model_eff = model_eff.to(device)
model_eff.eval()

print("✅ Best EfficientNet-B0 model loaded!")
print("Best validation accuracy: 87.87%")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np

model_eff.eval()

all_binary_predictions = []
all_binary_labels = []

all_lesion_predictions = []
all_lesion_labels = []

with torch.no_grad():

    for images, lesion_targets, binary_targets in test_loader:

        images = images.to(device)
        lesion_targets = lesion_targets.to(device)
        binary_targets = binary_targets.to(device)

        # Model prediction
        lesion_output, binary_output = model_eff(images)

        # Predictions
        lesion_pred = torch.argmax(lesion_output, dim=1)
        binary_pred = torch.argmax(binary_output, dim=1)

        # Store predictions
        all_lesion_predictions.extend(
            lesion_pred.cpu().numpy()
        )

        all_lesion_labels.extend(
            lesion_targets.cpu().numpy()
        )

        all_binary_predictions.extend(
            binary_pred.cpu().numpy()
        )

        all_binary_labels.extend(
            binary_targets.cpu().numpy()
        )


# ==========================================
# BINARY RESULTS
# ==========================================

binary_accuracy = accuracy_score(
    all_binary_labels,
    all_binary_predictions
)

binary_precision = precision_score(
    all_binary_labels,
    all_binary_predictions,
    average="weighted",
    zero_division=0
)

binary_recall = recall_score(
    all_binary_labels,
    all_binary_predictions,
    average="weighted",
    zero_division=0
)

binary_f1 = f1_score(
    all_binary_labels,
    all_binary_predictions,
    average="weighted",
    zero_division=0
)


print("=" * 50)
print("EFFICIENTNET-B0 - FINAL TEST RESULTS")
print("=" * 50)

print(f"Accuracy  : {binary_accuracy:.4f}")
print(f"Precision : {binary_precision:.4f}")
print(f"Recall    : {binary_recall:.4f}")
print(f"F1-Score  : {binary_f1:.4f}")


# ==========================================
# CONFUSION MATRIX
# ==========================================

cm = confusion_matrix(
    all_binary_labels,
    all_binary_predictions
)

print("\nConfusion Matrix:")
print(cm)


# ==========================================
# CLASSIFICATION REPORT
# ==========================================

print("\nClassification Report:")

print(
    classification_report(
        all_binary_labels,
        all_binary_predictions,
        target_names=["Benign", "Malignant"],
        zero_division=0
    )
)

In [ ]:
# 7-class lesion-type evaluation

lesion_accuracy = accuracy_score(
    all_lesion_labels,
    all_lesion_predictions
)

lesion_precision = precision_score(
    all_lesion_labels,
    all_lesion_predictions,
    average="weighted",
    zero_division=0
)

lesion_recall = recall_score(
    all_lesion_labels,
    all_lesion_predictions,
    average="weighted",
    zero_division=0
)

lesion_f1 = f1_score(
    all_lesion_labels,
    all_lesion_predictions,
    average="weighted",
    zero_division=0
)

print("=" * 50)
print("EFFICIENTNET-B0 - 7 CLASS LESION RESULTS")
print("=" * 50)

print(f"Accuracy  : {lesion_accuracy:.4f}")
print(f"Precision : {lesion_precision:.4f}")
print(f"Recall    : {lesion_recall:.4f}")
print(f"F1-Score  : {lesion_f1:.4f}")

print("\nClassification Report:")

print(
    classification_report(
        all_lesion_labels,
        all_lesion_predictions,
        target_names=class_names,
        zero_division=0
    )
)

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Transform recreated!")

predect

In [ ]:
import glob
import random
from PIL import Image
import matplotlib.pyplot as plt
import torch

# Pick one image from the corrected test set
sample = test_df.sample(1).iloc[0]

image_id = sample["image_id"]

# Find the image in HAM10000 folders
image_paths = glob.glob(
    f"/content/ham10000/**/{image_id}.jpg",
    recursive=True
)

if len(image_paths) == 0:
    raise FileNotFoundError(f"Image not found: {image_id}")

image_path = image_paths[0]

# Load image
image = Image.open(image_path).convert("RGB")

# Apply the same transform used by the test dataset
image_tensor = transform(image).unsqueeze(0).to(device)

# Prediction
model_eff.eval()

with torch.no_grad():
    lesion_output, binary_output = model_eff(image_tensor)

    lesion_probs = torch.softmax(lesion_output, dim=1)
    binary_probs = torch.softmax(binary_output, dim=1)

    lesion_pred = torch.argmax(lesion_probs, dim=1).item()
    binary_pred = torch.argmax(binary_probs, dim=1).item()

# Names
binary_names = ["Benign", "Malignant"]

predicted_binary = binary_names[binary_pred]
predicted_lesion = class_names[lesion_pred]

binary_confidence = binary_probs[0, binary_pred].item() * 100
lesion_confidence = lesion_probs[0, lesion_pred].item() * 100

actual_binary = binary_names[sample["binary_label"]]
actual_lesion = sample["dx"]

# Show image
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title(
    f"Prediction: {predicted_binary}\n"
    f"Lesion: {predicted_lesion}\n"
    f"Confidence: {binary_confidence:.2f}%"
)
plt.show()

# Print results
print("=" * 50)
print("EFFICIENTNET-B0 - INDIVIDUAL PREDICTION")
print("=" * 50)

print("Image ID            :", image_id)
print("Actual Diagnosis    :", actual_binary)
print("Actual Lesion       :", actual_lesion)

print("\nPredicted Diagnosis :", predicted_binary)
print("Binary Confidence   :", f"{binary_confidence:.2f}%")

print("Predicted Lesion    :", predicted_lesion)
print("Lesion Confidence   :", f"{lesion_confidence:.2f}%")

In [ ]:
# ============================================================
# EFFICIENTNET-B0 - EXTERNAL IMAGE PREDICTION
# Upload any skin lesion image and get:
# 1. Benign / Malignant
# 2. Lesion Type
# 3. Confidence
# ============================================================

import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

# ------------------------------------------------------------
# 1. Check GPU
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("EFFICIENTNET-B0 EXTERNAL IMAGE PREDICTION")
print("=" * 60)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. Class names
# ------------------------------------------------------------
class_names = [
    'akiec',
    'bcc',
    'bkl',
    'df',
    'mel',
    'nv',
    'vasc'
]

binary_names = [
    "Benign",
    "Malignant"
]

# ------------------------------------------------------------
# 3. Recreate EfficientNet-B0 model
# ------------------------------------------------------------
backbone_eff = models.efficientnet_b0(weights=None)

class MultiTaskEfficientNet(nn.Module):
    def __init__(self, backbone):
        super().__init__()

        self.features = backbone.features

        num_features = backbone.classifier[1].in_features

        self.lesion_head = nn.Linear(num_features, 7)
        self.binary_head = nn.Linear(num_features, 2)

    def forward(self, x):

        x = self.features(x)

        x = torch.nn.functional.adaptive_avg_pool2d(x, 1)

        x = torch.flatten(x, 1)

        lesion_output = self.lesion_head(x)
        binary_output = self.binary_head(x)

        return lesion_output, binary_output


model_eff = MultiTaskEfficientNet(backbone_eff).to(device)

# ------------------------------------------------------------
# 4. Load the BEST trained model
# ------------------------------------------------------------
model_path = "/content/drive/MyDrive/Skincancer_project/best_efficientnet_final.pth"

if not os.path.exists(model_path):
    raise FileNotFoundError(
        "Model checkpoint not found. Check your Google Drive path."
    )

checkpoint = torch.load(
    model_path,
    map_location=device
)

# Handle different checkpoint formats
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model_eff.load_state_dict(checkpoint["model_state_dict"])
else:
    model_eff.load_state_dict(checkpoint)

model_eff.eval()

print("\n✅ Best EfficientNet-B0 model loaded successfully!")

# ------------------------------------------------------------
# 5. Image preprocessing
# ------------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# 6. Upload external image
# ------------------------------------------------------------
print("\n📁 Please upload a skin lesion image...")

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No image uploaded.")

image_path = list(uploaded.keys())[0]

# ------------------------------------------------------------
# 7. Load image
# ------------------------------------------------------------
image = Image.open(image_path).convert("RGB")

# Convert image for model
image_tensor = transform(image).unsqueeze(0).to(device)

# ------------------------------------------------------------
# 8. Prediction
# ------------------------------------------------------------
with torch.no_grad():

    lesion_output, binary_output = model_eff(image_tensor)

    # Convert logits to probabilities
    lesion_probs = torch.softmax(lesion_output, dim=1)
    binary_probs = torch.softmax(binary_output, dim=1)

    # Get predicted class
    lesion_pred = torch.argmax(
        lesion_probs, dim=1
    ).item()

    binary_pred = torch.argmax(
        binary_probs, dim=1
    ).item()

# ------------------------------------------------------------
# 9. Get prediction names
# ------------------------------------------------------------
predicted_binary = binary_names[binary_pred]

predicted_lesion = class_names[lesion_pred]

# Confidence
binary_confidence = (
    binary_probs[0, binary_pred].item() * 100
)

lesion_confidence = (
    lesion_probs[0, lesion_pred].item() * 100
)

# ------------------------------------------------------------
# 10. Display image
# ------------------------------------------------------------
plt.figure(figsize=(7, 7))

plt.imshow(image)
plt.axis("off")

plt.title(
    f"Prediction: {predicted_binary}\n"
    f"Lesion Type: {predicted_lesion}\n"
    f"Binary Confidence: {binary_confidence:.2f}%"
)

plt.show()

# ------------------------------------------------------------
# 11. Print final result
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("PREDICTION RESULT")
print("=" * 60)

print("Image Name          :", image_path)

print("\n--- Binary Classification ---")
print("Predicted Diagnosis :", predicted_binary)
print("Confidence          :", f"{binary_confidence:.2f}%")

print("\n--- Lesion Classification ---")
print("Predicted Lesion    :", predicted_lesion)
print("Confidence          :", f"{lesion_confidence:.2f}%")

print("\n" + "=" * 60)
print("ALL 7 LESION CLASS PROBABILITIES")
print("=" * 60)

for i, name in enumerate(class_names):

    probability = lesion_probs[0, i].item() * 100

    print(f"{name:6s} : {probability:.2f}%")

print("\n" + "=" * 60)
print("BINARY CLASS PROBABILITIES")
print("=" * 60)

for i, name in enumerate(binary_names):

    probability = binary_probs[0, i].item() * 100

    print(f"{name:10s} : {probability:.2f}%")

print("\n" + "=" * 60)
print("VIVA EXPLANATION")
print("=" * 60)

print(
    f"The EfficientNet-B0 model classified the uploaded image "
    f"as {predicted_binary} with {binary_confidence:.2f}% "
    f"model confidence."
)

print(
    f"The predicted lesion type is {predicted_lesion} "
    f"with {lesion_confidence:.2f}% model confidence."
)

print(
    "\nNote: The confidence shown is the model's softmax "
    "confidence and should not be interpreted as a clinical diagnosis."
)

Densenet121

In [ ]:
# ============================================================
# DENSENET121 - IMAGE ONLY MODEL
# ============================================================

import torch
import torch.nn as nn
from torchvision import models
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# ------------------------------------------------------------
# 2. Load pretrained DenseNet121
# ------------------------------------------------------------
backbone_dense = models.densenet121(weights="DEFAULT")

# Number of features from DenseNet121
num_features = backbone_dense.classifier.in_features

# Remove original classifier
backbone_dense.classifier = nn.Identity()


# ------------------------------------------------------------
# 3. Multitask DenseNet121
# ------------------------------------------------------------
class MultiTaskDenseNet(nn.Module):

    def __init__(self, backbone):

        super().__init__()

        self.features = backbone.features

        self.lesion_head = nn.Linear(num_features, 7)

        self.binary_head = nn.Linear(num_features, 2)

    def forward(self, x):

        x = self.features(x)

        x = torch.relu(x)

        x = torch.nn.functional.adaptive_avg_pool2d(x, 1)

        x = torch.flatten(x, 1)

        lesion_output = self.lesion_head(x)

        binary_output = self.binary_head(x)

        return lesion_output, binary_output


model_dense = MultiTaskDenseNet(backbone_dense).to(device)

print("✅ DenseNet121 model created!")


# ------------------------------------------------------------
# 4. Calculate class weights
# ------------------------------------------------------------

# 7-class weights
lesion_classes = np.array([
    'akiec',
    'bcc',
    'bkl',
    'df',
    'mel',
    'nv',
    'vasc'
])

lesion_weights = compute_class_weight(
    class_weight="balanced",
    classes=lesion_classes,
    y=train_df["dx"]
)

lesion_weights = torch.tensor(
    lesion_weights,
    dtype=torch.float32
).to(device)


# Binary weights
binary_classes = np.array([0, 1])

binary_weights = compute_class_weight(
    class_weight="balanced",
    classes=binary_classes,
    y=train_df["binary_label"]
)

binary_weights = torch.tensor(
    binary_weights,
    dtype=torch.float32
).to(device)


print("\n7-Class Weights:")
for cls, weight in zip(lesion_classes, lesion_weights):
    print(cls, ":", round(weight.item(), 3))

print("\nBinary Weights:")
print("Benign   :", round(binary_weights[0].item(), 3))
print("Malignant:", round(binary_weights[1].item(), 3))


# ------------------------------------------------------------
# 5. Loss functions
# ------------------------------------------------------------
lesion_criterion_dense = nn.CrossEntropyLoss(
    weight=lesion_weights
)

binary_criterion_dense = nn.CrossEntropyLoss(
    weight=binary_weights
)


def multitask_loss_dense(
    lesion_output,
    binary_output,
    lesion_target,
    binary_target
):

    lesion_loss = lesion_criterion_dense(
        lesion_output,
        lesion_target
    )

    binary_loss = binary_criterion_dense(
        binary_output,
        binary_target
    )

    total_loss = lesion_loss + binary_loss

    return total_loss, lesion_loss, binary_loss


# ------------------------------------------------------------
# 6. Optimizer
# ------------------------------------------------------------
optimizer_dense = torch.optim.Adam(
    model_dense.parameters(),
    lr=0.0001,
    weight_decay=1e-4
)

print("\n✅ Optimizer ready!")

# ------------------------------------------------------------
# 7. Training settings
# ------------------------------------------------------------
num_epochs = 10

best_val_acc = 0.0

best_model_path = (
    "/content/drive/MyDrive/Skincancer_project/"
    "best_densenet_final.pth"
)

print("\nTraining DenseNet121...")
print("=" * 60)


# ------------------------------------------------------------
# 8. Training loop
# ------------------------------------------------------------
for epoch in range(num_epochs):

    # -------------------------
    # Training
    # -------------------------
    model_dense.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, lesion_labels, binary_labels in train_loader:

        images = images.to(device)

        lesion_labels = lesion_labels.to(device)

        binary_labels = binary_labels.to(device)

        optimizer_dense.zero_grad()

        lesion_output, binary_output = model_dense(images)

        loss, _, _ = multitask_loss_dense(
            lesion_output,
            binary_output,
            lesion_labels,
            binary_labels
        )

        loss.backward()

        optimizer_dense.step()

        train_loss += loss.item()

        binary_pred = torch.argmax(
            binary_output,
            dim=1
        )

        train_correct += (
            binary_pred == binary_labels
        ).sum().item()

        train_total += binary_labels.size(0)

    train_loss /= len(train_loader)

    train_acc = train_correct / train_total


    # -------------------------
    # Validation
    # -------------------------
    model_dense.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, lesion_labels, binary_labels in val_loader:

            images = images.to(device)

            lesion_labels = lesion_labels.to(device)

            binary_labels = binary_labels.to(device)

            lesion_output, binary_output = model_dense(
                images
            )

            loss, _, _ = multitask_loss_dense(
                lesion_output,
                binary_output,
                lesion_labels,
                binary_labels
            )

            val_loss += loss.item()

            binary_pred = torch.argmax(
                binary_output,
                dim=1
            )

            val_correct += (
                binary_pred == binary_labels
            ).sum().item()

            val_total += binary_labels.size(0)

    val_loss /= len(val_loader)

    val_acc = val_correct / val_total


    # -------------------------
    # Print
    # -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Binary Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Binary Acc: {val_acc:.4f}"
    )


    # -------------------------
    # Save best model
    # -------------------------
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model_dense.state_dict(),
            best_model_path
        )

        print("✅ New best DenseNet121 model saved!")


print("\n" + "=" * 60)
print("DENSENET121 TRAINING COMPLETED")
print("=" * 60)

print(
    f"Best Validation Binary Accuracy: "
    f"{best_val_acc * 100:.2f}%"
)

print("Model saved at:")
print(best_model_path)

In [ ]:
# ============================================================
# DENSENET121 - FINAL TEST EVALUATION
# Corrected lesion-level test set
# ============================================================

import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("DENSENET121 - FINAL TEST EVALUATION")
print("=" * 60)
print("Device:", device)

# ------------------------------------------------------------
# 2. Recreate DenseNet121 architecture
# ------------------------------------------------------------
backbone_dense = models.densenet121(weights=None)

num_features = backbone_dense.classifier.in_features

backbone_dense.classifier = nn.Identity()


class MultiTaskDenseNet(nn.Module):

    def __init__(self, backbone):

        super().__init__()

        self.features = backbone.features

        self.lesion_head = nn.Linear(num_features, 7)

        self.binary_head = nn.Linear(num_features, 2)

    def forward(self, x):

        x = self.features(x)

        x = torch.relu(x)

        x = torch.nn.functional.adaptive_avg_pool2d(x, 1)

        x = torch.flatten(x, 1)

        lesion_output = self.lesion_head(x)

        binary_output = self.binary_head(x)

        return lesion_output, binary_output


model_dense = MultiTaskDenseNet(backbone_dense).to(device)

# ------------------------------------------------------------
# 3. Load BEST DenseNet121 model
# ------------------------------------------------------------
model_path = (
    "/content/drive/MyDrive/Skincancer_project/"
    "best_densenet_final.pth"
)

checkpoint = torch.load(
    model_path,
    map_location=device
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model_dense.load_state_dict(
        checkpoint["model_state_dict"]
    )
else:
    model_dense.load_state_dict(checkpoint)

model_dense.eval()

print("✅ Best DenseNet121 model loaded!")

# ------------------------------------------------------------
# 4. Class names
# ------------------------------------------------------------
class_names = [
    'akiec',
    'bcc',
    'bkl',
    'df',
    'mel',
    'nv',
    'vasc'
]

binary_names = [
    "Benign",
    "Malignant"
]

# ------------------------------------------------------------
# 5. Predictions on TEST SET
# ------------------------------------------------------------
all_binary_true = []
all_binary_pred = []

all_lesion_true = []
all_lesion_pred = []

with torch.no_grad():

    for images, lesion_labels, binary_labels in test_loader:

        images = images.to(device)

        lesion_labels = lesion_labels.to(device)

        binary_labels = binary_labels.to(device)

        lesion_output, binary_output = model_dense(images)

        # Binary prediction
        binary_pred = torch.argmax(
            binary_output,
            dim=1
        )

        # Lesion prediction
        lesion_pred = torch.argmax(
            lesion_output,
            dim=1
        )

        all_binary_true.extend(
            binary_labels.cpu().numpy()
        )

        all_binary_pred.extend(
            binary_pred.cpu().numpy()
        )

        all_lesion_true.extend(
            lesion_labels.cpu().numpy()
        )

        all_lesion_pred.extend(
            lesion_pred.cpu().numpy()
        )


# ------------------------------------------------------------
# 6. Binary metrics
# ------------------------------------------------------------
binary_accuracy = accuracy_score(
    all_binary_true,
    all_binary_pred
)

binary_precision = precision_score(
    all_binary_true,
    all_binary_pred,
    average="weighted",
    zero_division=0
)

binary_recall = recall_score(
    all_binary_true,
    all_binary_pred,
    average="weighted",
    zero_division=0
)

binary_f1 = f1_score(
    all_binary_true,
    all_binary_pred,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 60)
print("BINARY CLASSIFICATION RESULTS")
print("=" * 60)

print(f"Accuracy  : {binary_accuracy:.4f}")
print(f"Precision : {binary_precision:.4f}")
print(f"Recall    : {binary_recall:.4f}")
print(f"F1-Score  : {binary_f1:.4f}")


# ------------------------------------------------------------
# 7. Binary classification report
# ------------------------------------------------------------
print("\nBinary Classification Report:")
print(
    classification_report(
        all_binary_true,
        all_binary_pred,
        target_names=binary_names,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 8. Binary confusion matrix
# ------------------------------------------------------------
cm_binary = confusion_matrix(
    all_binary_true,
    all_binary_pred
)

print("Binary Confusion Matrix:")
print(cm_binary)


plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_binary,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=binary_names,
    yticklabels=binary_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("DenseNet121 - Binary Confusion Matrix")

plt.show()


# ------------------------------------------------------------
# 9. 7-Class lesion metrics
# ------------------------------------------------------------
lesion_accuracy = accuracy_score(
    all_lesion_true,
    all_lesion_pred
)

lesion_precision = precision_score(
    all_lesion_true,
    all_lesion_pred,
    average="weighted",
    zero_division=0
)

lesion_recall = recall_score(
    all_lesion_true,
    all_lesion_pred,
    average="weighted",
    zero_division=0
)

lesion_f1 = f1_score(
    all_lesion_true,
    all_lesion_pred,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 60)
print("7-CLASS LESION CLASSIFICATION RESULTS")
print("=" * 60)

print(f"Accuracy  : {lesion_accuracy:.4f}")
print(f"Precision : {lesion_precision:.4f}")
print(f"Recall    : {lesion_recall:.4f}")
print(f"F1-Score  : {lesion_f1:.4f}")


# ------------------------------------------------------------
# 10. Lesion classification report
# ------------------------------------------------------------
print("\nLesion Classification Report:")

print(
    classification_report(
        all_lesion_true,
        all_lesion_pred,
        target_names=class_names,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 11. Lesion confusion matrix
# ------------------------------------------------------------
cm_lesion = confusion_matrix(
    all_lesion_true,
    all_lesion_pred
)

print("7-Class Confusion Matrix:")
print(cm_lesion)


plt.figure(figsize=(8, 7))

sns.heatmap(
    cm_lesion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("DenseNet121 - 7-Class Lesion Confusion Matrix")

plt.show()


# ------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("DENSENET121 FINAL TEST SUMMARY")
print("=" * 60)

print(
    f"Binary Accuracy : {binary_accuracy * 100:.2f}%"
)

print(
    f"Binary F1       : {binary_f1 * 100:.2f}%"
)

print(
    f"Lesion Accuracy : {lesion_accuracy * 100:.2f}%"
)

print(
    f"Lesion F1       : {lesion_f1 * 100:.2f}%"
)

In [ ]:
# ============================================================
# DENSENET121 - EXTERNAL IMAGE PREDICTION
# Upload any skin lesion image
# ============================================================

import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("DENSENET121 - EXTERNAL IMAGE PREDICTION")
print("=" * 60)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. Class names
# ------------------------------------------------------------
class_names = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc"
]

binary_names = [
    "Benign",
    "Malignant"
]


# ------------------------------------------------------------
# 3. Recreate DenseNet121 architecture
# ------------------------------------------------------------
backbone_dense = models.densenet121(weights=None)

num_features = backbone_dense.classifier.in_features

backbone_dense.classifier = nn.Identity()


class MultiTaskDenseNet(nn.Module):

    def __init__(self, backbone):

        super().__init__()

        self.features = backbone.features

        self.lesion_head = nn.Linear(
            num_features, 7
        )

        self.binary_head = nn.Linear(
            num_features, 2
        )

    def forward(self, x):

        x = self.features(x)

        x = torch.relu(x)

        x = torch.nn.functional.adaptive_avg_pool2d(
            x, 1
        )

        x = torch.flatten(x, 1)

        lesion_output = self.lesion_head(x)

        binary_output = self.binary_head(x)

        return lesion_output, binary_output


model_dense = MultiTaskDenseNet(
    backbone_dense
).to(device)


# ------------------------------------------------------------
# 4. Load BEST DenseNet121 model
# ------------------------------------------------------------
model_path = (
    "/content/drive/MyDrive/Skincancer_project/"
    "best_densenet_final.pth"
)

if not os.path.exists(model_path):

    raise FileNotFoundError(
        "DenseNet121 model checkpoint not found."
    )


checkpoint = torch.load(
    model_path,
    map_location=device
)


if (
    isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
):

    model_dense.load_state_dict(
        checkpoint["model_state_dict"]
    )

else:

    model_dense.load_state_dict(
        checkpoint
    )


model_dense.eval()

print("\n✅ Best DenseNet121 model loaded!")


# ------------------------------------------------------------
# 5. Image preprocessing
# ------------------------------------------------------------
transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# 6. Upload external image
# ------------------------------------------------------------
print("\n📁 Upload a skin lesion image:")

uploaded = files.upload()

if len(uploaded) == 0:

    raise ValueError(
        "No image was uploaded."
    )


image_path = list(uploaded.keys())[0]


# ------------------------------------------------------------
# 7. Load image
# ------------------------------------------------------------
image = Image.open(
    image_path
).convert("RGB")


image_tensor = transform(
    image
).unsqueeze(0).to(device)


# ------------------------------------------------------------
# 8. Prediction
# ------------------------------------------------------------
with torch.no_grad():

    lesion_output, binary_output = model_dense(
        image_tensor
    )

    # Convert outputs to probabilities
    lesion_probs = torch.softmax(
        lesion_output,
        dim=1
    )

    binary_probs = torch.softmax(
        binary_output,
        dim=1
    )

    # Predicted classes
    lesion_pred = torch.argmax(
        lesion_probs,
        dim=1
    ).item()

    binary_pred = torch.argmax(
        binary_probs,
        dim=1
    ).item()


# ------------------------------------------------------------
# 9. Prediction names
# ------------------------------------------------------------
predicted_binary = binary_names[
    binary_pred
]

predicted_lesion = class_names[
    lesion_pred
]


# ------------------------------------------------------------
# 10. Confidence
# ------------------------------------------------------------
binary_confidence = (
    binary_probs[0, binary_pred].item()
    * 100
)

lesion_confidence = (
    lesion_probs[0, lesion_pred].item()
    * 100
)


# ------------------------------------------------------------
# 11. Display image
# ------------------------------------------------------------
plt.figure(figsize=(7, 7))

plt.imshow(image)

plt.axis("off")

plt.title(
    f"Prediction: {predicted_binary}\n"
    f"Lesion Type: {predicted_lesion}\n"
    f"Confidence: {binary_confidence:.2f}%"
)

plt.show()


# ------------------------------------------------------------
# 12. Final prediction
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("DENSENET121 - PREDICTION RESULT")
print("=" * 60)

print("Image Name          :", image_path)

print("\n--- Binary Classification ---")

print(
    "Predicted Diagnosis :",
    predicted_binary
)

print(
    "Binary Confidence   :",
    f"{binary_confidence:.2f}%"
)

print("\n--- Lesion Classification ---")

print(
    "Predicted Lesion    :",
    predicted_lesion
)

print(
    "Lesion Confidence   :",
    f"{lesion_confidence:.2f}%"
)


# ------------------------------------------------------------
# 13. All 7 lesion probabilities
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("7-CLASS LESION PROBABILITIES")
print("=" * 60)

for i, name in enumerate(class_names):

    probability = (
        lesion_probs[0, i].item()
        * 100
    )

    print(
        f"{name:6s} : {probability:.2f}%"
    )


# ------------------------------------------------------------
# 14. Binary probabilities
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("BINARY CLASS PROBABILITIES")
print("=" * 60)

for i, name in enumerate(binary_names):

    probability = (
        binary_probs[0, i].item()
        * 100
    )

    print(
        f"{name:10s} : {probability:.2f}%"
    )


# ------------------------------------------------------------
# 15. Viva explanation
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("VIVA EXPLANATION")
print("=" * 60)

print(
    f"The DenseNet121 model classified the uploaded "
    f"image as {predicted_binary} with "
    f"{binary_confidence:.2f}% model confidence."
)

print(
    f"The predicted lesion type is "
    f"{predicted_lesion} with "
    f"{lesion_confidence:.2f}% model confidence."
)

print(
    "\nNote: The confidence is the model's softmax "
    "confidence and should not be considered a "
    "clinical diagnosis."
)

Mutimodel Image+Age+Gender

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("⚠️ GPU is NOT available")

Traning for MultiModel

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = [
    "EfficientNet-B0",
    "DenseNet121",
    "MM EfficientNet-B0",
    "MM DenseNet121"
]

accuracy = [87.80, 87.67, 89.34, 88.47]
f1_score = [88.03, 87.94, 89.43, 88.75]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(12, 6))

bars1 = plt.bar(x - width/2, accuracy, width, label="Accuracy")
bars2 = plt.bar(x + width/2, f1_score, width, label="F1-Score")

plt.xlabel("Models")
plt.ylabel("Performance (%)")
plt.title("Binary Classification Performance Comparison")

plt.xticks(x, models, rotation=15)
plt.ylim(75, 95)

plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.3)

# Add values on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.2,
            f"{height:.2f}%",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

# Save image
plt.savefig(
    "/content/binary_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
import matplotlib.pyplot as plt
import numpy as np

models = [
    "EfficientNet-B0",
    "DenseNet121",
    "MM EfficientNet-B0",
    "MM DenseNet121"
]

accuracy = [81.50, 82.44, 81.43, 84.12]
f1_score = [82.07, 81.05, 82.17, 84.57]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(12, 6))

bars1 = plt.bar(x - width/2, accuracy, width, label="Accuracy")
bars2 = plt.bar(x + width/2, f1_score, width, label="F1-Score")

plt.xlabel("Models")
plt.ylabel("Performance (%)")
plt.title("7-Class Lesion Classification Performance Comparison")

plt.xticks(x, models, rotation=15)
plt.ylim(75, 90)

plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.3)

# Add values on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.2,
            f"{height:.2f}%",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

# Save image
plt.savefig(
    "/content/lesion_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = [
    "EfficientNet-B0",
    "DenseNet121",
    "MM EfficientNet-B0",
    "MM DenseNet121"
]

accuracy = [81.50, 82.44, 81.43, 84.12]
f1_score = [82.07, 81.05, 82.17, 84.57]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(12, 6))

bars1 = plt.bar(x - width/2, accuracy, width, label="Accuracy")
bars2 = plt.bar(x + width/2, f1_score, width, label="F1-Score")

plt.xlabel("Models")
plt.ylabel("Performance (%)")
plt.title("7-Class Lesion Classification Performance Comparison")

plt.xticks(x, models, rotation=15)
plt.ylim(75, 90)

plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.3)

# Add values on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.2,
            f"{height:.2f}%",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

# Save image
plt.savefig(
    "/content/lesion_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = [
    "EfficientNet-B0",
    "DenseNet121",
    "MM EfficientNet-B0",
    "MM DenseNet121"
]

binary_accuracy = [87.80, 87.67, 89.34, 88.47]
lesion_accuracy = [81.50, 82.44, 81.43, 84.12]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(13, 7))

bars1 = plt.bar(
    x - width/2,
    binary_accuracy,
    width,
    label="Binary Accuracy"
)

bars2 = plt.bar(
    x + width/2,
    lesion_accuracy,
    width,
    label="7-Class Lesion Accuracy"
)

plt.xlabel("Models", fontsize=12)
plt.ylabel("Accuracy (%)", fontsize=12)

plt.title(
    "Performance Comparison of Skin Lesion Classification Models",
    fontsize=15,
    fontweight="bold"
)

plt.xticks(x, models, rotation=15)
plt.ylim(75, 95)

plt.legend(fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.3)

# Add values on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.25,
            f"{height:.2f}%",
            ha="center",
            va="bottom",
            fontsize=10
        )

plt.tight_layout()

# Save high-quality image
plt.savefig(
    "/content/combined_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Confusion matrices
cm_efficientnet = np.array([
    [1099, 106],
    [76, 211]
])

cm_densenet = np.array([
    [1099, 106],
    [66, 221]
])

cm_mm_efficientnet = np.array([
    [1119, 86],
    [73, 214]
])

cm_mm_densenet = np.array([
    [1099, 106],
    [66, 221]
])

matrices = [
    ("EfficientNet-B0", cm_efficientnet),
    ("DenseNet121", cm_densenet),
    ("MM EfficientNet-B0", cm_mm_efficientnet),
    ("MM DenseNet121", cm_mm_densenet)
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (title, cm) in zip(axes.ravel(), matrices):

    # Blue color map
    im = ax.imshow(cm, cmap="Blues")

    # Add values
    for i in range(2):
        for j in range(2):
            ax.text(
                j, i,
                str(cm[i, j]),
                ha="center",
                va="center",
                fontsize=16,
                fontweight="bold"
            )

    ax.set_title(title, fontsize=13, fontweight="bold")

    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("Actual Label", fontsize=11)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["Benign", "Malignant"])
    ax.set_yticklabels(["Benign", "Malignant"])

    # Add color bar for each matrix
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    "Confusion Matrices – Binary Classification",
    fontsize=17,
    fontweight="bold"
)

plt.tight_layout()

# Save for PPT
plt.savefig(
    "/content/confusion_matrices_blue.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()